# Multi-Source AI Agent for Bangladesh Institution, Hospital, and Restaurant Queries

This notebook demonstrates an AI agent that answers questions about Bangladesh using three local SQLite databases and live web search.

## Architecture
| Tool | Data Source | Covers |
|---|---|---|
| `InstitutionsDBTool` | `institutions.db` | Schools, madrashas, colleges, technical institutes |
| `HospitalsDBTool` | `hospitals.db` | Hospitals, clinics, health facilities |
| `RestaurantsDBTool` | `restaurants.db` | Restaurants, ratings, reviews |
| `WebSearchTool` | Tavily API (live web) | General knowledge, policies, anything not in the DBs |

The agent uses LLaMA 3.3 70B (via Groq) to:
1. Read the user's question and pick the right tool
2. Convert the question to SQL (for DB tools)
3. Turn raw SQL results back into a natural language answer

## Imports and Environment Setup

In [1]:
import os
from dotenv import load_dotenv
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from tools.institutions_tool import InstitutionsDBTool
from tools.hospitals_tool import HospitalsDBTool
from tools.restaurants_tool import RestaurantsDBTool
from tools.web_search_tool import WebSearchTool

# Load GROQ_API_KEY and TAVILY_API_KEY from .env file
load_dotenv()

print("All imports successful")
print(f"GROQ_API_KEY   : {'set' if os.environ.get('GROQ_API_KEY') else 'MISSING'}")
print(f"TAVILY_API_KEY : {'set' if os.environ.get('TAVILY_API_KEY') else 'MISSING'}")

All imports successful
GROQ_API_KEY   : set
TAVILY_API_KEY : set


## Build the Agent

In [2]:
# LLM - LLaMA 3.3 70B on Groq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Register all four tools
tools = [
    InstitutionsDBTool(),
    HospitalsDBTool(),
    RestaurantsDBTool(),
    WebSearchTool(),
]

# System prompt - tells the agent when to use which tool
system_prompt = """You are an AI agent that answers questions about Bangladesh.

You have access to four tools:
1. InstitutionsDBTool for questions about schools, madrashas, colleges, and technical institutes, including their location and education level
2. HospitalsDBTool for questions about hospitals and other health facilities, including whether they are private or government and what type of facility they are. There is no bed count or doctor count data.
3. RestaurantsDBTool for questions about restaurants, ratings, and reviews. There is no cuisine type data and location is matched from free text addresses.
4. WebSearchTool for general knowledge questions not covered by the three databases above, and for any question asking about data the databases do not contain, such as bed capacity or cuisine type

Always pick the single most relevant tool for the question.
If a question mixes both a data lookup and a general knowledge part, use both tools and combine the answers.
If a question asks for data that a database tool does not contain, say so plainly instead of guessing.
Give your final answer in clear and simple natural language."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Agent is ready")

Agent is ready


## Helper Function

A small wrapper so each demo question is one clean line.

In [3]:
def ask(question: str) -> str:
    """Send a question to the agent and print a formatted result."""
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print('='*60)
    response = agent_executor.invoke({"input": question})
    answer = response["output"]
    print(f"\nAnswer: {answer}")
    return answer

---
## Demo 1 - Institutions Database

Questions about schools, madrashas, colleges, and technical institutes.

In [4]:
ask("How many madrashas are there in Sylhet division?")


Question: How many madrashas are there in Sylhet division?


> Entering new AgentExecutor chain...

Invoking: `InstitutionsDBTool` with `{'query': 'madrashas in Sylhet division'}`


There are numerous madrashas in the Sylhet division, with a significant number located in the Habiganj district. These madrashas offer various levels of education, including Dakhil, Alim, and Fazil, and are managed by non-government organizations. Some of the madrashas are co-educational, while others are exclusively for girls. The madrashas are recognized by the relevant authorities and receive MPO status, indicating that they receive government funding.
Invoking: `InstitutionsDBTool` with `{'query': 'number of madrashas in Sylhet division'}`


There are 400 madrashas in Sylhet division.There are 400 madrashas in Sylhet division.

> Finished chain.

Answer: There are 400 madrashas in Sylhet division.


'There are 400 madrashas in Sylhet division.'

In [5]:
ask("Which district has the most government colleges?")


Question: Which district has the most government colleges?


> Entering new AgentExecutor chain...

Invoking: `InstitutionsDBTool` with `{'query': 'district with most government colleges'}`


The district with the most government colleges is Dhaka.Dhaka district has the most government colleges.

> Finished chain.

Answer: Dhaka district has the most government colleges.


'Dhaka district has the most government colleges.'

---
## Demo 2 - Hospitals Database

Questions about hospitals, clinics, and health facilities.

In [6]:
ask("How many private hospitals are in Dhaka?")


Question: How many private hospitals are in Dhaka?


> Entering new AgentExecutor chain...

Invoking: `HospitalsDBTool` with `{'query': 'number of private hospitals in Dhaka'}`


There are 5684 private hospitals in Dhaka.There are 5684 private hospitals in Dhaka.

> Finished chain.

Answer: There are 5684 private hospitals in Dhaka.


'There are 5684 private hospitals in Dhaka.'

In [7]:
ask("List private hospitals in Chattogram.")


Question: List private hospitals in Chattogram.


> Entering new AgentExecutor chain...

Invoking: `HospitalsDBTool` with `{'query': 'private hospitals in Chattogram'}`


There are several private hospitals in Chattogram, including BGC Trust Medical College, Central Medical College, and Chottagram Ma O Shishu Medical College, among others. Additionally, there are various NGO hospitals and clinics in the division, such as those operated by BRAC (BHP) in different upazilas of Brahmanbaria district.There are several private hospitals in Chattogram, including BGC Trust Medical College, Central Medical College, and Chottagram Ma O Shishu Medical College, among others. Additionally, there are various NGO hospitals and clinics in the division, such as those operated by BRAC (BHP) in different upazilas of Brahmanbaria district.

> Finished chain.

Answer: There are several private hospitals in Chattogram, including BGC Trust Medical College, Central Medical College, and Chottagram Ma O Shishu

'There are several private hospitals in Chattogram, including BGC Trust Medical College, Central Medical College, and Chottagram Ma O Shishu Medical College, among others. Additionally, there are various NGO hospitals and clinics in the division, such as those operated by BRAC (BHP) in different upazilas of Brahmanbaria district.'

---
## Demo 3 - Restaurants Database

Questions about restaurants, ratings, and reviews.

In [8]:
ask("What are the top 3 highest rated restaurants in Dhaka?")


Question: What are the top 3 highest rated restaurants in Dhaka?


> Entering new AgentExecutor chain...

Invoking: `RestaurantsDBTool` with `{'query': 'top 3 highest rated restaurants in Dhaka'}`


The top 3 highest rated restaurants in Dhaka are শুভ এন্টারপ্রাইজ অলটাইম ডিলার, ভাত ঘর, and আপন নিবাস, all with a perfect 5.0 rating.The top 3 highest rated restaurants in Dhaka are শুভ এন্টারপ্রাইজ অলটাইম ডিলার, ভাত ঘর, and আপন নিবাস, all with a perfect 5.0 rating.

> Finished chain.

Answer: The top 3 highest rated restaurants in Dhaka are শুভ এন্টারপ্রাইজ অলটাইম ডিলার, ভাত ঘর, and আপন নিবাস, all with a perfect 5.0 rating.


'The top 3 highest rated restaurants in Dhaka are শুভ এন্টারপ্রাইজ অলটাইম ডিলার, ভাত ঘর, and আপন নিবাস, all with a perfect 5.0 rating.'

In [9]:
ask("Which restaurant in Chattogram has the most reviews?")


Question: Which restaurant in Chattogram has the most reviews?


> Entering new AgentExecutor chain...

Invoking: `RestaurantsDBTool` with `{'query': 'Chattogram restaurant with the most reviews'}`


The restaurant with the most reviews in Chattogram is Hotel Noorjahan, with 17,655 reviews.The restaurant with the most reviews in Chattogram is Hotel Noorjahan, with 17,655 reviews.

> Finished chain.

Answer: The restaurant with the most reviews in Chattogram is Hotel Noorjahan, with 17,655 reviews.


'The restaurant with the most reviews in Chattogram is Hotel Noorjahan, with 17,655 reviews.'

---
## Demo 4 - Web Search Tool

General knowledge questions not covered by the local databases.

In [10]:
ask("What is the current education policy for primary schools in Bangladesh?")


Question: What is the current education policy for primary schools in Bangladesh?


> Entering new AgentExecutor chain...

Invoking: `WebSearchTool` with `{'query': 'current education policy for primary schools in Bangladesh'}`


[PDF] National Education Policy-English corrected _2_: Primary education will be universal, compulsory, free and of uniform quality for all. At present 100% children cannot be given access to primary schools

Education in Bangladesh - Wikipedia: The Ministry of Primary and Mass Education implements policies for primary education education in Bangladesh is compulsory for all citizens until the end of

Bangladesh, pre-primary education and the school learning ...: The entry age of the current one-year pre-primary education initiative in Bangladesh is 5. This age limit is adhered to by government providers of education

Extending primary education for our future #NewsUpdate ...: We applaud the government's decision to extend free primary school education up to g

'The current education policy for primary schools in Bangladesh is that primary education will be universal, compulsory, free, and of uniform quality for all. The Ministry of Primary and Mass Education implements policies for primary education, and education in Bangladesh is compulsory for all citizens until the end of primary school. The government has also extended free primary school education up to grade eight, in line with the National Education Policy.'

In [11]:
ask("What is the national healthcare budget of Bangladesh in 2024?")


Question: What is the national healthcare budget of Bangladesh in 2024?


> Entering new AgentExecutor chain...

Invoking: `WebSearchTool` with `{'query': 'Bangladesh national healthcare budget 2024'}`


Bangladesh Budget: FY-2024-25 Health sector budget increases by ...: FY-2024-25 Health sector budget increases by 8.81% from FY. Bangladesh has allocated 5.19 percent of the proposed national budget. BDT 7, 97,

[PDF] Health Budget of Bangladesh - Centre for Policy Dialogue (CPD): In FY2010, 41 per cent of the budget was directed towards non-development expenditures 3 Figure 2: Budget Allocation for Health Source: Authors’ illustration based on data from the Ministry of Finance (MOF, 2024a). BDT in thousand crore Per cent 0 10 20 30 40 50 60 0.0 5.0 10.0 15.0 20.0 25.0 30.0 FY10 FY11 FY12 FY13 FY14 FY15 FY16 FY17 FY18 FY19 FY20 FY21 FY22 FY23 Non-development Development T otal Development expenditure as percentage of total Figure 3: Revised Budget Allocation for Health Source: Authors

'The national healthcare budget of Bangladesh in 2024 is BDT 419 billion, which is approximately 5.19% of the proposed national budget.'

---
## Demo 5 - Multi-Tool Routing

Questions that require more than one tool.

In [12]:
ask("How many hospitals are in Cox's Bazar, and what kind of tourists visit the area?")


Question: How many hospitals are in Cox's Bazar, and what kind of tourists visit the area?


> Entering new AgentExecutor chain...

Invoking: `HospitalsDBTool` with `{'query': "hospitals in Cox's Bazar"}`


There are numerous hospitals and health centers in Cox's Bazar, including the Cox's Bazar 250 Bed District Sadar Hospital, Cox's Bazar Chest Disease Clinic, and several Upazila Health Complexes such as Chakaria, Kutubdia, Moheshkhali, and Pekua. Additionally, there are many Union Health Sub Centers throughout the district, providing healthcare services to local communities.
Invoking: `WebSearchTool` with `{'query': "Cox's Bazar tourist information"}`


Cox's Bazar Bangladesh 🇧🇩 | Cox’s Bazar Tourist Spot | Cox’s Bazar Travel Guide: Cox's Bazar Bangladesh 🇧🇩 | Cox’s Bazar Tourist Spot | Cox’s Bazar Travel Guide
Suurojit Palmal
47100 subscribers
105 likes
5172 views
10 Dec 2023
Hi
My name is Surojit Palmal. I am from West Bengal. Welcome to MY YouTube Channel Surojit Palmal.

About t

"There are numerous hospitals and health centers in Cox's Bazar, including the Cox's Bazar 250 Bed District Sadar Hospital, Cox's Bazar Chest Disease Clinic, and several Upazila Health Complexes such as Chakaria, Kutubdia, Moheshkhali, and Pekua. Additionally, there are many Union Health Sub Centers throughout the district, providing healthcare services to local communities. Cox's Bazar is a popular tourist destination in Bangladesh, known for its natural beauty, including the world's longest natural sea beach, and various attractions such as Himchori, Kutubdia Lighthouse, Safari park, Moheshkhali Island, and St. Martin Island. The area attracts tourists who come to enjoy the scenic beauty, relax on the beach, and explore the local culture."